# 03. MCP Client/Server 기본 구현: stdio 서버와 클라이언트는 무엇을 주고받는가

이 노트북의 질문은 하나입니다. **에이전트가 도구를 부를 때, 그 사이에 오가는 것은 정확히 무엇인가?**
이번 시간에는 요구사항 인터뷰 MCP 서버 한 개(도구 7개)를 세 가지 방법으로 만져 봅니다.

1. 모델 없이, 메시지 한 줄 한 줄을 직접 보내고 받는다 (`mcp.Client`, 그리고 그 아래의 원시 JSON 줄).
2. 같은 서버를 `langchain.mcp.MCPAdapter`로 붙여 LangChain 도구로 바꾼다.
3. 그 도구를 Deep Agent에 주고, 모델이 도구를 고르는 모습을 한 번만 본다 (유일한 유료 셀).

모든 코드는 제공됩니다. 실행하고, 출력을 읽고, 바꿔 보는 것이 이번 시간의 일입니다. 점수는 없습니다.

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 1 | stdio: 서버는 자식 프로세스다 (준비와 환경 확인 포함) | 4 |
| 2 | 선로 위의 메시지: handshake 없는 요청, `_meta`, 결정적 목록 순서 | 9 |
| 3 | 한 세션 걷기: draft, ready, frozen 과 두 종류의 오류 | 11 |
| 4 | stdout 오염: 서버가 print 한 줄로 망가지는 과정 | 6 |
| 5 | MCPAdapter: MCP 도구가 LangChain 도구가 되는 과정 | 6 |
| 6 | Deep Agent 한 턴 (유료 1회) | 5 |
| 7 | 직접 해보기, 확인 질문, 정리 | 9 |

합계 50분입니다.

## 호출 규모

- 구간 1~5는 **무료**입니다. 모델도, Jev도, 외부 API도 부르지 않습니다. 내 컴퓨터에서 자식 프로세스(서버)만 띄웁니다.
- 구간 6의 셀 하나가 **Claude 약 3~6회**를 씁니다(상한 6회로 안내합니다). Jev 호출은 없습니다.
- 직접 해보기의 유료 셀 둘은 `RUN_EXERCISE = False`, `RUN_OPTIONAL = False`로 잠겨 있습니다. 직접 켜기 전에는 아무것도 호출하지 않습니다.
- 유료 셀은 실행 직전에 `announce(...)`가 규모를 한 줄로 알려 줍니다. 크레딧은 본인의 것입니다.

## 이번 시간이 끝나면 할 수 있는 것

- stdio 전송에서 stdout 과 stderr 의 역할을 구분하고, 어긴 서버가 어떻게 실패하는지 설명한다.
- `tools/list`, `tools/call` 한 번이 선로에서 어떤 JSON 줄로 보이는지 말한다.
- 업무 오류(`success: false`)와 도구 오류(`isError: true`)를 결과 모양으로 구분한다.
- MCP 도구가 LangChain 도구로 바뀔 때 이름, 인자 스키마, enum, annotations 가 어떻게 남는지 확인한다.

## 1. stdio: 서버는 자식 프로세스다

stdio 전송에서 MCP 서버는 클라이언트가 띄우는 **자식 프로세스**입니다. 클라이언트는 서버 프로그램의 표준 입력(stdin)에
요청을 한 줄씩 쓰고, 표준 출력(stdout)에서 응답을 한 줄씩 읽습니다. 별도의 포트도, 네트워크도 없습니다.

여기서 가장 중요한 규칙은 하나입니다. **stdout 은 프로토콜 메시지만 싣는 통로입니다.**
사람이 읽을 로그, 디버그 출력, 경고는 모두 **stderr** 로 보내야 합니다. 이 저장소의 서버도 그렇게 합니다.
`cap_lab/server/server.py:435` 의 주석이 이 규칙을 적어 둔 곳입니다.

```python
# stdio의 stdout은 MCP 통신에 사용한다. 진단 로그는 stderr에 쓴다.
server.run(transport="stdio")
```

이 서버는 `cap_lab/server/server.py:177` 의 `MCPServer(name="requirements-lab", ...)` 로 만들어지고,
도구 7개를 `publish(...)` 데코레이터로 등록합니다(`start_interview` 는 `:253`, `freeze_seed` 는 `:326`).
이 노트북의 모든 셀은 같은 서버 파일을 매번 새 자식 프로세스로 띄웁니다. 세션 DB 와 근거(Evidence) 파일은
`Day04_new/outputs/notebook/<이름>-<날짜-시각>/` 아래에 쌓입니다(`cap_lab/notebook.py:69` 의 `new_workdir`).

먼저 이 노트북이 쓰는 도구를 가져옵니다.

In [ ]:
# 준비: 이 노트북 전체에서 쓰는 가져오기. 모델도 외부 API도 부르지 않는다.
import asyncio
import itertools
import json
import os
import subprocess
import sys
import tempfile
import threading
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

from fastmcp.client.transports import PythonStdioTransport
from langchain.mcp import MCPAdapter
from mcp import Client
from mcp.client.stdio import StdioServerParameters

from cap_lab.agent import SERVER_SCRIPT, SKILLS_DIR, build_agent, server_argv
from cap_lab.experiment import load_cases
from cap_lab.notebook import announce, connect_server, key_table, new_workdir, show_events, show_table
from cap_lab.server.server import FaultPlan
from cap_lab.settings import ROOT, load_settings

# 시험 서버마다 작업 폴더 이름이 겹치지 않게 붙이는 번호.
PING_NUMBER = itertools.count(1)

# 서버 7개 도구의 이름. 서버가 등록한 순서다(cap_lab/server/server.py의 publish 순서).
TOOL_ORDER = ["start_interview", "record_answer", "get_session", "freeze_seed",
              "list_evidence", "read_evidence", "save_evidence"]
print("준비 완료. 작업 폴더 기준:", Path.cwd().name)

In [ ]:
# [따라하기] 설치된 버전, 키 설정 상태, 서버 스크립트 위치를 확인한다.
# 1) 이 노트북이 설명하는 동작은 아래 버전에서 확인한 것이다. 버전이 다르면 출력이 다를 수 있다.
for package in ("mcp", "fastmcp", "langchain", "deepagents", "langchain-anthropic"):
    print(f"{package:20s} {version(package)}")

# 2) 서버는 자식 프로세스로 뜨는 파이썬 파일이다. 어디에 있는지 본다.
print("서버 스크립트:", SERVER_SCRIPT.relative_to(ROOT))
print("서버를 띄우는 인터프리터:", Path(sys.executable).name)

# 3) 키는 설정됨/미설정만 보여 준다. 값은 어디에도 출력하지 않는다.
#    구간 6의 유료 셀만 ANTHROPIC_API_KEY가 필요하다. 구간 1~5는 키 없이 돈다.
key_table()

## 2. 선로 위의 메시지: handshake 없는 요청

MCP 2026-07-28 판은 연결마다 **initialize 로 상태를 합의하던 방식을 없앴습니다**(`docs/DESIGN.md` 5절의 SEP-2575).
대신 **요청마다** 프로토콜 버전과 클라이언트 정보, 클라이언트 capabilities 를 `params._meta` 에 실어 보냅니다.
서버는 요청 하나만 보고 처리할 수 있으므로 연결에 상태가 없습니다(stateless). 서버 정보가 궁금하면
`server/discover` 요청을 보냅니다.

말로 들으면 추상적이므로 직접 확인합니다. 아래 도우미는 서버를 `subprocess`로 띄워서
**JSON 한 줄을 stdin 에 쓰고, stdout 한 줄을 읽는 일만** 합니다. SDK(`mcp.Client`)는 쓰지 않습니다.
`_meta` 에 들어가는 세 키는 `.venv/lib/python3.13/site-packages/mcp_types/_types.py:53-62` 에 상수로 정의돼 있습니다.

In [ ]:
# 도우미: SDK 없이 JSON 줄을 직접 주고받는다. 서버가 응답하지 않을 때를 대비해 30초 뒤 강제 종료한다.
RAW_META = {
    "io.modelcontextprotocol/protocolVersion": "2026-07-28",
    "io.modelcontextprotocol/clientInfo": {"name": "raw-notebook", "version": "0"},
    "io.modelcontextprotocol/clientCapabilities": {},
}


def raw_exchange(messages: list[dict], *, timeout_s: float = 30.0) -> tuple[list[str], str]:
    """새 서버 프로세스를 띄워 messages를 한 줄씩 보내고, 응답 줄 목록과 stderr 전체를 돌려준다.

    handshake(initialize)를 하지 않는다. 요청 하나를 쓰고 응답 한 줄을 읽은 뒤 다음 요청을 쓴다.
    """
    with tempfile.TemporaryDirectory() as scratch:
        scratch_path = Path(scratch)
        argv = server_argv(scratch_path / "sessions.sqlite3", scratch_path / "evidence")
        # 서버에는 안전한 변수(PATH, HOME)만 물려준다. 키를 넘기지 않는다.
        env = {name: os.environ[name] for name in ("PATH", "HOME") if name in os.environ}
        server = subprocess.Popen(
            [sys.executable, str(SERVER_SCRIPT), *argv],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            text=True, encoding="utf-8", cwd=str(ROOT), env=env,
        )
        # 응답이 영영 오지 않으면 readline이 멈추므로, 시간이 지나면 프로세스를 죽여 읽기를 풀어 준다.
        watchdog = threading.Timer(timeout_s, server.kill)
        watchdog.start()
        responses = []
        try:
            for message in messages:
                server.stdin.write(json.dumps(message, ensure_ascii=False) + "\n")
                server.stdin.flush()
                responses.append(server.stdout.readline().rstrip("\n"))
        finally:
            watchdog.cancel()
            server.kill()
            stderr_text = server.stderr.read()
            server.wait()
    return responses, stderr_text


def brief(line: str, limit: int = 160) -> str:
    """응답 JSON 줄을 읽기 쉽게 줄인다. 빈 줄은 서버가 응답하지 않았다는 뜻이다."""
    return line[:limit] + (" ..." if len(line) > limit else "") if line else "(응답 없음)"

**먼저 예측하세요.** 서버를 막 띄웠고 아무 인사도 나누지 않았습니다(`initialize` 도, `server/discover` 도 없음).
이 상태에서 `_meta` 를 담은 `tools/list` 요청 한 줄을 곧바로 보내면 서버는 (1) 도구 목록을 돌려준다,
(2) 오류를 돌려준다, (3) 아무 응답이 없다 중 무엇일까요? 이유와 함께 적어 두고 실행하세요.

In [ ]:
# [따라하기] handshake 없이 tools/list 한 줄을 보낸다.
# 1) 요청 한 줄. params._meta 에 버전, 클라이언트 정보, capabilities 가 들어 있다.
list_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {"_meta": RAW_META}}

# 2) 새 서버 프로세스에 보내고 응답 한 줄을 받는다.
responses, stderr_text = raw_exchange([list_request])
reply = json.loads(responses[0])

# 3) 응답의 모양과 도구 이름 순서를 본다.
print("요청:", json.dumps(list_request, ensure_ascii=False)[:150], "...")
print("응답의 최상위 키:", sorted(reply))
print("result 의 키:", sorted(reply["result"]))
print("resultType:", reply["result"]["resultType"], "| cacheScope:", reply["result"]["cacheScope"])
print("도구 이름:", [tool["name"] for tool in reply["result"]["tools"]])
print("stderr 로 나온 것:", repr(stderr_text) if stderr_text else "(없음)")

이제 같은 요청에서 `_meta` 만 빼 봅니다. 서버가 요청을 어떻게 읽는지 알려면 요청을 일부러 어겨 보는 것이 가장 빠릅니다.

**먼저 예측하세요.** `params` 를 빈 객체로 두면 어떻게 될까요? 서버가 "버전을 모르니 기본값으로 처리한다"고
너그럽게 넘어갈까요, 거절할까요? 거절한다면 오류 코드는 무엇일 것 같나요?

In [ ]:
# [깨뜨리기] _meta 없이 tools/list 를 보낸다. 새 서버 프로세스이므로 앞 셀의 연결과 아무 관계가 없다.
broken_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}

responses, stderr_text = raw_exchange([broken_request])
error_reply = json.loads(responses[0])

print("보낸 요청:", json.dumps(broken_request, ensure_ascii=False))
print("받은 응답:", brief(responses[0]))
print("오류 코드:", error_reply.get("error", {}).get("code"))
print("오류 메시지:", error_reply.get("error", {}).get("message"))
print("오류 data:", repr(error_reply.get("error", {}).get("data")))

**관찰할 것.** 앞 셀은 도구 목록을 돌려주었고, 이 셀은 `error` 객체를 돌려주었습니다. 작성 환경에서는 오류 코드가
`-32602`(Invalid params), 메시지가 `Invalid request parameters`, `data` 는 빈 문자열이었습니다.
관찰한 사실은 여기까지입니다. 서버가 `_meta` 를 **왜** 필수로 보는지는 이 노트북이 코드로 확인하지 않았습니다.
다만 두 셀을 나란히 놓으면 결론은 분명합니다. **stateless 요청에서 `_meta` 는 선택이 아니라 요청의 일부입니다.**
그리고 오류의 `data` 가 비어 있으므로, 오류 메시지만으로는 무엇이 빠졌는지 알 수 없습니다.
직접 JSON 을 쓸 일은 드물지만, SDK 를 쓰다 이런 오류를 만나면 가장 먼저 `_meta` 를 의심하세요.

이번에는 SDK 가 같은 일을 하는 모습을 선로에서 직접 봅니다. 자식 프로세스와 클라이언트 **사이**에 끼어서
오가는 줄을 파일에 적어 주는 작은 중계 스크립트를 쓰겠습니다. 서버 코드는 한 줄도 바꾸지 않습니다.

In [ ]:
# 도우미: 클라이언트와 서버 사이에서 줄을 그대로 전달하며 파일에 적는 중계 스크립트를 작업 폴더에 만든다.
WIRE_TEE_SOURCE = '''\
import json, subprocess, sys, threading
# 사용법: wire_tee.py <기록 파일> <서버 명령...>
log_path, command = sys.argv[1], sys.argv[2:]
log = open(log_path, "a", encoding="utf-8")
server = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=sys.stderr)


def pump(source, sink, direction):
    for line in iter(source.readline, b""):
        log.write(json.dumps({"direction": direction, "line": line.decode("utf-8").rstrip("\\n")}, ensure_ascii=False) + "\\n")
        log.flush()
        sink.write(line)
        sink.flush()
    if sink is server.stdin:
        sink.close()


threading.Thread(target=pump, args=(sys.stdin.buffer, server.stdin, "client->server"), daemon=True).start()
pump(server.stdout, sys.stdout.buffer, "server->client")
server.wait()
'''


def wire_rows(log_path: Path) -> list[dict]:
    """중계 기록(JSON 줄)을 표의 행으로 바꾼다. 요청은 method 와 _meta 키, 응답은 결과 모양을 보여 준다."""
    rows = []
    for number, raw in enumerate(log_path.read_text(encoding="utf-8").splitlines(), start=1):
        entry = json.loads(raw)
        message = json.loads(entry["line"])
        if "method" in message:
            params = message.get("params") or {}
            meta_keys = sorted(key.split("/")[-1] for key in params.get("_meta", {}))
            kind, detail = message["method"], f"_meta 키: {meta_keys}"
        else:
            result = message.get("result", {})
            kind = "오류" if "error" in message else "결과"
            detail = f"resultType={result.get('resultType')}, cacheScope={result.get('cacheScope')}, ttlMs={result.get('ttlMs')}"
        rows.append({"순서": number, "방향": entry["direction"], "id": message.get("id"),
                     "종류": kind, "내용": detail, "바이트": len(entry["line"].encode("utf-8"))})
    return rows

In [ ]:
# [따라하기] mcp.Client 가 서버와 주고받는 줄을 중계 스크립트로 가로채서 표로 본다.
# 1) 작업 폴더와 중계 스크립트, 기록 파일 경로를 만든다.
wire_dir = new_workdir("nb03-wire")
tee_script = wire_dir / "wire_tee.py"
tee_script.write_text(WIRE_TEE_SOURCE, encoding="utf-8")
wire_log = wire_dir / "wire.jsonl"

# 2) 클라이언트가 띄우는 명령을 '중계 스크립트 + 진짜 서버'로 바꾼다.
real_server = [sys.executable, str(SERVER_SCRIPT),
               *server_argv(wire_dir / "sessions.sqlite3", wire_dir / "evidence")]
wire_params = StdioServerParameters(
    command=sys.executable, args=[str(tee_script), str(wire_log), *real_server], cwd=str(ROOT)
)

# 3) 평소처럼 연결해서 tools/list 한 번, tools/call 한 번을 한다.
async with Client(wire_params) as wire_client:
    print("협상된 프로토콜 버전:", wire_client.protocol_version)
    listed = await wire_client.list_tools()
    started = await wire_client.call_tool("start_interview", {"goal": "선로 관찰용 목표"})
    print("도구 수:", len(listed.tools), "| start_interview 성공 여부:", started.structured_content["success"])

# 4) 선로에 오간 줄을 표로 본다.
show_table(wire_rows(wire_log))

**관찰할 것.** 표에서 세 가지를 확인하세요.

1. **첫 줄이 `server/discover` 입니다.** `initialize` 라는 이름의 요청은 한 번도 없습니다. `mcp.Client` 의 `mode` 기본값이 `auto`
   이며 "discover 를 시도하고, 옛 서버면 initialize 로 되돌아간다"고 `.venv/lib/python3.13/site-packages/mcp/client/client.py:335-338` 에 적혀 있습니다.
2. **클라이언트가 보내는 모든 줄에 `_meta` 가 있습니다.** `protocolVersion`, `clientInfo`, `clientCapabilities` 세 키입니다.
   앞서 손으로 쓴 요청과 같은 모양입니다. SDK 가 대신 붙여 줄 뿐입니다.
3. **결과마다 `resultType`, `cacheScope` 가 붙어 있습니다.** `tools/list` 결과의 `ttlMs` 는 이 서버에서 0, `cacheScope` 는 `private` 이었습니다
   (캐시를 어떻게 쓰는지는 04 노트북의 주제입니다). 이 서버가 캐시를 켜지 않았다는 뜻으로만 읽으세요.

메시지 한 개가 한 줄이라는 점도 눈여겨보세요. 이것이 다음 구간(stdout 오염)의 전제입니다.

In [ ]:
# [따라하기] 연결 직후 클라이언트가 알고 있는 서버 정보와, tools/list 의 순서가 일정한지 본다.
async def listed_names(label: str) -> list[str]:
    """새 연결을 열어 도구 이름을 서버가 돌려준 순서대로 모은다."""
    async with connect_server(new_workdir(label)) as client:
        return [tool.name for tool in (await client.list_tools()).tools]


# 1) 같은 서버에 서로 다른 연결을 두 번 열어 순서를 비교한다.
first_names = await listed_names("nb03-order-a")
second_names = await listed_names("nb03-order-b")

# 2) 연결 직후 클라이언트가 가진 서버 정보.
async with connect_server(new_workdir("nb03-info")) as info_client:
    print("프로토콜 버전:", info_client.protocol_version)
    print("서버 이름:", info_client.server_info.name)
    print("서버 capabilities 중 tools:", info_client.server_capabilities.tools)

print("첫 번째 연결의 순서:", first_names)
print("두 연결의 순서가 같은가:", first_names == second_names)
print("알파벳순 정렬과 같은가:", first_names == sorted(first_names))
print("서버가 등록한 순서와 같은가:", first_names == TOOL_ORDER)

**관찰할 것.** 두 연결의 순서가 같았고, 알파벳순이 **아니었으며**, 서버 파일에서 등록한 순서와 같았습니다.
2026-07-28 판은 `tools/list` 의 순서를 결정적으로 하라고 요구합니다(`docs/DESIGN.md` 5절). 이 노트북이 확인한 것은
"이 서버에서 연결 두 번이 같은 순서였다"는 것까지입니다. 모든 서버가 그렇다는 증명은 아닙니다.
순서가 일정해야 하는 이유는 모델에 도구 목록을 넣는 순서가 프롬프트의 일부이기 때문입니다.
순서가 흔들리면 같은 입력에서도 요청 앞부분이 달라져 캐시와 재현이 어려워집니다.

## 3. 한 세션 걷기: draft, ready, frozen

이 서버는 요구사항 인터뷰를 도구 7개로 나눠 놓았습니다. 세션은 세 상태 중 하나입니다.
상태는 저장된 기록(이벤트)을 처음부터 다시 읽어서 정해집니다(`cap_lab/server/workflow.py:82`).

```python
status = "frozen" if seed is not None else "draft" if missing else "ready"
```

- `draft`: 필수 질문 두 개(`constraints`, `acceptance_criteria`) 중 답이 없는 것이 있다.
- `ready`: 두 답이 모두 모였다. 아직 확정은 아니다.
- `frozen`: `freeze_seed` 가 명세(Seed)를 확정했다. 이후 답변은 바꿀 수 없다.

아래에서 모델 없이 도구를 **순서대로 직접** 부릅니다. 모델이 나중에 해야 할 일을 사람이 먼저 해 보는 것입니다.
먼저 호출 결과를 한 줄로 요약해 주는 도우미를 둡니다.

In [ ]:
# 도우미: 도구 한 번을 부르고 결과를 한 줄로 요약해 출력한다. 원본 결과는 그대로 돌려준다.
async def step(client: Client, label: str, tool_name: str, arguments: dict) -> dict:
    """tools/call 한 번. 성공이면 상태와 남은 질문, 업무 오류이면 오류 코드를 보여 준다."""
    result = await client.call_tool(tool_name, arguments)
    structured = result.structured_content or {}
    if result.is_error:
        # 도구 오류(isError): 결과에 structuredContent 가 없고 본문 텍스트에 오류가 있다.
        summary = f"도구 오류(isError=True): {result.content[0].text.splitlines()[0]}"
    elif structured.get("success") is False:
        # 업무 오류: tools/call 은 성공했고, 결과 안에 success=false 와 오류 코드가 있다.
        summary = f"업무 오류: error={structured['error']!r} / {structured['message']}"
    else:
        summary = (f"status={structured.get('status')} | 남은 질문={structured.get('missing_fields')} "
                   f"| 기록 수={structured.get('event_count')}")
    print(f"[{label}] {tool_name}\n    isError={result.is_error} -> {summary}")
    return structured

**먼저 예측하세요.** 목표만 말한 직후(`start_interview`), 제약만 답한 직후(`record_answer` 한 번),
완료 기준까지 답한 직후(`record_answer` 두 번), 확정 직후(`freeze_seed`)의 `status` 는 각각 무엇일까요?
그리고 `event_count`(저장된 기록 수)는 몇이 될까요? 네 값을 먼저 적고 실행하세요.

In [ ]:
# [따라하기] 한 세션을 처음부터 근거 저장까지 걷는다. 모델은 없다. 사람이 도구를 차례로 부른다.
GOAL = "회의실 예약이 잡히면 참석자에게 알림 메일을 보내는 기능"
ANSWERS = {
    "constraints": "없음",
    "acceptance_criteria": "예약 10건을 만들면 알림 메일 10통이 발송 기록에 남는다",
}

walk_dir = new_workdir("nb03-walk")
status_history = []

async with connect_server(walk_dir) as client:
    # 1) 목표를 말한다 -> 세션이 생기고 draft 가 된다.
    started = await step(client, "1. 목표", "start_interview", {"goal": GOAL})
    session_id = started["session_id"]
    status_history.append(started["status"])

    # 2) 필수 질문에 하나씩 답한다. 서버가 돌려준 next_question.field 를 그대로 field 로 쓴다.
    for field, answer in ANSWERS.items():
        recorded = await step(client, f"2. 답변 {field}", "record_answer",
                              {"session_id": session_id, "field": field, "answer": answer})
        status_history.append(recorded["status"])

    # 3) 세션을 다시 조회한다. 읽기처럼 보이지만 근거 파일을 복구하는 도구다(cap_lab/server/server.py:312-314 주석).
    await step(client, "3. 조회", "get_session", {"session_id": session_id})

    # 4) 확정한다 -> frozen, 그리고 명세(seed)와 그 근거 ID 가 생긴다.
    frozen = await step(client, "4. 확정", "freeze_seed", {"session_id": session_id})
    status_history.append(frozen["status"])
    seed_evidence_id = frozen["seed_evidence_id"]

    # 5) 작성한 요약을 근거로 저장한다. 원본 근거(seed)의 ID 를 source_evidence_id 로 연결한다.
    saved = await step(client, "5. 저장", "save_evidence", {
        "session_id": session_id, "title": "인터뷰 요약", "source_evidence_id": seed_evidence_id,
        "content": f"목표: {GOAL}\n제약: {ANSWERS['constraints']}\n완료 기준: {ANSWERS['acceptance_criteria']}",
    })

    # 6) 이 세션의 근거 목록을 본다. 종류별로 몇 개인가.
    listed_evidence = await client.call_tool("list_evidence", {"session_id": session_id})
    kinds = Counter(item["kind"] for item in listed_evidence.structured_content["evidence"])

print("\n상태 변화:", " -> ".join(status_history))
print("저장된 산출물 ID:", saved["evidence"]["evidence_id"][:20] + "...")
print("근거 종류별 개수:", dict(kinds))

**관찰할 것.** 상태가 `draft -> draft -> ready -> frozen` 으로 움직였습니다. 첫 `draft` 는 목표 직후, 둘째 `draft` 는
제약만 답한 직후, `ready` 는 완료 기준까지 답한 직후입니다. 각 호출의 `isError` 는 모두 `False` 입니다.
그리고 `list_evidence` 의 종류별 개수에는 사람이 부른 적 없는 `tool_call` 이 섞여 있습니다. 서버 미들웨어
`record_tools`(`cap_lab/server/server.py:155`)가 **모든 `tools/call` 을 감사 기록으로 남기기** 때문입니다.
이 기록은 도구의 효과가 아니라 "누가 무엇을 불렀는가"의 기록입니다.

In [ ]:
# [따라하기] 서버가 근거 폴더에 남긴 이벤트를 세션별로 본다. 파일은 도구 결과를 돌려주기 전에 쓰인다.
show_events(walk_dir / "evidence")

# 같은 폴더의 파일 이름 접두사별 개수. event-* 는 상태를 만드는 기록, seed-* 는 확정 명세,
# artifact-* 는 save_evidence 로 저장한 산출물, call-* 는 호출 감사 기록이다.
session_folder = walk_dir / "evidence" / session_id
prefixes = Counter(path.name.split("-")[0] for path in session_folder.glob("*.json"))
print("파일 접두사별 개수:", dict(prefixes))

### 오류에는 두 종류가 있습니다

위에서 본 `isError=False` 는 "도구가 실행되었다"는 뜻입니다. 도구가 **업무적으로** 거절해도 `tools/call` 자체는 성공입니다.
서버는 업무 오류를 결과 안의 `success: false` 와 오류 코드로 돌려줍니다(`cap_lab/server/workflow.py:27-32` 의 `WorkflowError`).
반대로 도구에 **잘못된 인자**가 들어오면 SDK 가 도구를 실행하기 전에 막고 `isError: true` 로 답합니다.
이 구분은 모델이 오류에서 회복할 수 있는지를 가릅니다. 두 종류를 각각 깨뜨려 봅니다.

**먼저 예측하세요.** 새 세션에서 아무 질문에도 답하지 않고 곧바로 `freeze_seed` 를 부르면 `isError` 는 `True` 일까요, `False` 일까요?
오류 코드와, 모델이 이어서 할 수 있는 일을 위해 결과에 들어 있을 만한 정보는 무엇일까요?

In [ ]:
# [깨뜨리기] 업무 오류: ready 가 아닌 세션을 확정하려 하고, 확정된 세션을 고치려 한다.
break_dir = new_workdir("nb03-break")
async with connect_server(break_dir) as client:
    fresh = await step(client, "새 세션", "start_interview", {"goal": "깨뜨리기 연습용 목표"})
    fresh_id = fresh["session_id"]

    # 1) draft 인 세션을 확정하려 한다.
    too_early = await step(client, "너무 이른 확정", "freeze_seed", {"session_id": fresh_id})
    print("    결과 전체:", json.dumps(too_early, ensure_ascii=False))

    # 2) 두 답을 채우고 확정한 뒤, 확정된 세션의 답을 바꾸려 한다.
    for field, answer in ANSWERS.items():
        await client.call_tool("record_answer", {"session_id": fresh_id, "field": field, "answer": answer})
    await step(client, "정상 확정", "freeze_seed", {"session_id": fresh_id})
    too_late = await step(client, "확정 뒤 수정", "record_answer",
                          {"session_id": fresh_id, "field": "constraints", "answer": "바꾸고 싶다"})
    print("    결과 전체:", json.dumps(too_late, ensure_ascii=False))

In [ ]:
# [깨뜨리기] 도구 오류: 스키마가 허용하지 않는 field 값을 보낸다. 서버의 업무 코드까지 가지 못한다.
async with connect_server(break_dir) as client:
    # 앞 셀과 다른 서버 프로세스지만 같은 작업 폴더(같은 DB)를 쓴다. 세션 ID 도 그대로 쓸 수 있다.
    wrong_field = await client.call_tool(
        "record_answer", {"session_id": fresh_id, "field": "budget", "answer": "무엇이든"}
    )

print("isError:", wrong_field.is_error)
print("structuredContent:", wrong_field.structured_content)
print("본문 텍스트(앞 3줄):")
for line in wrong_field.content[0].text.splitlines()[:3]:
    print("   ", line)

**관찰할 것.** 두 셀의 결과 모양이 다릅니다.

| 구분 | `isError` | `structuredContent` | 오류가 담긴 곳 | 모델이 할 수 있는 일 |
|---|---|---|---|---|
| 업무 오류 (`seed_incomplete`, `seed_frozen`) | `False` | `success: false`, `error`, `message` (+ `missing_fields`) | 결과의 JSON | 오류 코드를 읽고 다음 행동을 고른다 (남은 질문에 답하거나 새 세션을 시작) |
| 도구 오류 (enum 위반) | `True` | 없음 | 본문 텍스트 | 오류 문장을 읽고 인자를 고쳐 다시 부른다 |

`seed_incomplete` 에는 `missing_fields` 가 함께 있어서 "무엇이 비었는가"를 모델이 구조로 읽을 수 있습니다(`cap_lab/server/workflow.py:222-227`).
enum 위반은 pydantic 검증 오류 문장이 그대로 본문에 담깁니다. 서버의 `invoke` 함수가 업무 오류만 값으로 바꾸고
나머지 예외는 SDK 의 도구 오류로 넘기는 구조입니다(`cap_lab/server/server.py:202-208`).
다만 enum 위반은 서버에 **업무 코드가 실행되기 전에** 막혔다는 점도 보세요. 같은 호출은 `record_tools` 미들웨어에는 기록됩니다
(`cap_lab/server/server.py:158` 의 주석: 입력 검증 전부터 감싼다).

In [ ]:
# [바꿔보기] 같은 호출을 반복하면 어떻게 되는가: start_interview(idempotent=False)와 record_answer(idempotent=True).
# 아래 TARGET_TOOL 을 "start_interview" 로 바꿔 두 결과를 비교해 보자.
TARGET_TOOL = "record_answer"

repeat_dir = new_workdir("nb03-repeat")
async with connect_server(repeat_dir) as client:
    base = await client.call_tool("start_interview", {"goal": "반복 호출 관찰용 목표"})
    repeat_session = base.structured_content["session_id"]
    arguments = ({"session_id": repeat_session, "field": "constraints", "answer": "없음"}
                 if TARGET_TOOL == "record_answer" else {"goal": "반복 호출 관찰용 목표"})

    # 같은 호출을 세 번 보낸다.
    outcomes = []
    for attempt in (1, 2, 3):
        result = await client.call_tool(TARGET_TOOL, arguments)
        outcomes.append((attempt, result.structured_content.get("session_id", "")[:8],
                         result.structured_content.get("event_count")))
    # 서버가 가진 세션 목록(세션 ID 생략 시 세션 목록을 돌려준다)
    sessions = (await client.call_tool("list_evidence", {})).structured_content["sessions"]

print(f"{TARGET_TOOL} 를 같은 인자로 3번 호출:")
for attempt, session_prefix, count in outcomes:
    print(f"  {attempt}번째: 세션={session_prefix} 기록 수={count}")
print("서버에 있는 세션 수:", len(sessions))

**관찰할 것.** `record_answer` 를 같은 인자로 세 번 보내도 기록 수는 늘지 않았고 세션은 하나였습니다.
`TARGET_TOOL = "start_interview"` 로 바꾸면 호출마다 **새 세션**이 생겨 세션 수가 늘어납니다.
서버는 이 차이를 도구 주석(annotations)의 `idempotentHint` 로 알려 줍니다(`cap_lab/server/server.py:251`, `:307`).
다만 annotations 는 클라이언트에게 주는 **힌트**일 뿐 서버가 강제하는 규칙이 아닙니다. 실제 동작은 업무 코드가 정합니다.
응답을 놓쳐서 다시 부르면 안 되는 도구가 무엇인지는, 04 노트북에서 재시도와 함께 다시 만납니다.

## 4. stdout 오염: print 한 줄이 서버를 망가뜨린다

1구간에서 stdout 은 "메시지 한 줄 = JSON 한 개"가 오가는 통로라고 했습니다. 서버 개발자가 디버깅하려고
`print("서버 시작합니다!")` 를 한 줄 넣으면 어떻게 될까요? 이 구간에서는 아주 작은 MCP 서버 두 개를
작업 폴더에 직접 만들어서 확인합니다. 둘 다 도구 `ping` 하나만 가지고, **차이는 print 한 줄의 목적지뿐**입니다.

클라이언트가 stdout 한 줄을 JSON 으로 읽지 못하면 어떻게 하는지는 `.venv/lib/python3.13/site-packages/mcp/client/stdio.py:219-226` 의 `_parse_line` 이 정합니다.
오류를 로그에 남기고, 그 오류를 값으로 세션에 넘깁니다.

In [ ]:
# 도우미: 서버 스크립트 두 개를 작업 폴더에 만들고, 클라이언트가 ping 을 부를 수 있는지 시험하는 함수.
QUIET_SERVER = '''\
import sys
from mcp.server import MCPServer

server = MCPServer(name="quiet")


@server.tool()
def ping() -> str:
    "응답한다."
    return "pong"


print("서버 시작합니다!", file=sys.stderr)   # 진단 로그는 stderr 로
server.run(transport="stdio")
'''

# print 의 끝 문자와 flush 만 다르게 만든 서버 두 개. 둘 다 시작할 때 stdout 으로 한 번 출력한다.
NOISY_TEMPLATE = QUIET_SERVER.replace(
    'print("서버 시작합니다!", file=sys.stderr)   # 진단 로그는 stderr 로',
    'print("서버 시작합니다!", {options})   # 진단 로그를 stdout 으로 (잘못된 예)',
)
NOISY_VARIANTS = {
    "줄바꿈 있는 print": NOISY_TEMPLATE.format(options="flush=True"),
    "줄바꿈 없는 print": NOISY_TEMPLATE.format(options='end="", flush=True'),
}


def leaf_errors(error: BaseException) -> list[BaseException]:
    """ExceptionGroup 안쪽에 겹겹이 싸인 실제 예외들을 평평하게 꺼낸다."""
    if isinstance(error, BaseExceptionGroup):
        return [leaf for inner in error.exceptions for leaf in leaf_errors(inner)]
    return [error]


async def try_ping(label: str, script_source: str, *, read_timeout_s: float = 5.0) -> None:
    """script_source 를 파일로 저장해 서버로 띄우고 ping 을 부른다. 성공/실패와 걸린 시간을 출력한다."""
    script_path = new_workdir(f"nb03-ping{next(PING_NUMBER)}") / "server.py"
    script_path.write_text(script_source, encoding="utf-8")
    # mode 를 프로토콜 버전으로 고정해 server/discover 단계를 건너뛴다. 첫 응답이 곧 ping 의 응답이 된다.
    client = Client(StdioServerParameters(command=sys.executable, args=[str(script_path)]),
                    mode="2026-07-28", read_timeout_seconds=read_timeout_s)
    started_at = time.perf_counter()
    try:
        async with client:
            result = await client.call_tool("ping", {})
        outcome = f"성공: {result.content[0].text!r}"
    except BaseException as error:  # 실패 원인을 보려고 일부러 넓게 잡는다
        leaves = leaf_errors(error)
        outcome = "실패: " + "; ".join(f"{type(leaf).__name__}: {leaf}" for leaf in leaves)
    print(f"[{label}] {time.perf_counter() - started_at:.1f}초 -> {outcome}")

In [ ]:
# [따라하기] 대조군: 같은 print 를 stderr 로 보낸 서버. 클라이언트는 문제없이 ping 을 부른다.
await try_ping("stderr 로 print", QUIET_SERVER)

**먼저 예측하세요.** 아래 셀은 print 를 stdout 으로 보내는 서버 두 개를 시험합니다. 하나는 print 가 줄바꿈으로 끝나고,
하나는 줄바꿈 없이(`end=""`) 끝납니다. 각각 (1) 문제없이 성공한다, (2) 오류 로그는 남지만 성공한다,
(3) 실패한다 중 무엇일까요? 클라이언트가 "읽지 못한 줄"을 어떻게 다룬다고 생각하는지 근거와 함께 적어 두세요.
읽기 대기는 5초로 제한되어 있습니다.

In [ ]:
# [깨뜨리기] print 를 stdout 으로 보내는 서버 두 개. 두 번째는 줄바꿈이 없어 서버의 첫 응답 앞에 글자가 붙는다.
# 셀의 stderr 출력에 "Failed to parse JSONRPC message from server" 로그와 파이썬 추적이 나타난다.
for variant_label, variant_source in NOISY_VARIANTS.items():
    await try_ping(variant_label, variant_source)

**관찰할 것.** 작성 환경에서의 **1회 관찰**입니다. 두 서버 모두 stderr 에 `Failed to parse JSONRPC message from server` 로그가 남았습니다.

- **줄바꿈 있는 print**: 그 줄이 독립된 한 줄이므로 클라이언트가 그 줄만 읽지 못한 줄로 처리하고 넘어갔고, `ping` 은 성공했습니다.
  "경고가 떴는데 되니까 괜찮다"로 보이는 가장 위험한 경우입니다.
- **줄바꿈 없는 print**: 글자가 서버의 **첫 응답 줄 앞에 붙어서** 그 응답 전체가 JSON 이 아닌 줄이 되었습니다.
  클라이언트는 응답을 영영 받지 못하고 `tools/call` 이 5초 뒤 시간 초과(`MCPError`, 코드 `-32001`)로 끝났습니다.
  실패한 쪽은 서버 코드의 오류가 아니라 **응답이 사라진 것**입니다. 그래서 원인이 stdout 이라는 단서가 `MCPError` 에는 없고,
  stderr 의 구문 분석 로그에만 있습니다.

`mode` 를 프로토콜 버전으로 고정한 이유도 알아 두세요. 기본값 `auto` 에서는 첫 요청인 `server/discover` 의 응답이 깨졌을 때
클라이언트가 다른 경로로 되돌아가 **느리게 성공하기도** 합니다(작성자는 약 10초가 걸려 성공하는 경우를 1회 관찰했습니다).
느리게 성공하는 것은 더 알아채기 어려운 실패입니다. 규칙은 간단합니다. **stdio 서버에서 진단 출력은 stderr 로만 보냅니다.**
이 저장소의 서버가 `cap_lab/server/server.py:435` 에 주석을 남긴 이유입니다.

## 5. MCPAdapter: MCP 도구가 LangChain 도구가 되는 과정

지금까지는 사람이 `mcp.Client` 로 도구를 불렀습니다. 에이전트는 LangChain/LangGraph 의 도구 객체를 받습니다.
`langchain.mcp.MCPAdapter` 가 둘을 잇는 다리입니다. `cap_lab/agent.py:110` 의 `build_agent` 가 바로 이 방식으로 같은 서버를 붙입니다.

```python
adapter = MCPAdapter(PythonStdioTransport(SERVER_SCRIPT, args=server_args, env=..., cwd=str(ROOT)))
tools = await adapter.list_tools()
```

`adapter.list_tools()` 는 서버의 `tools/list` 결과를 LangChain `StructuredTool` 목록으로 바꿉니다.
모델이 보게 되는 것은 이름, 설명, 인자 스키마입니다. 앞에서 `mcp.Client` 로 본 것이 어디까지 남는지 표로 확인합니다.

In [ ]:
# [따라하기] 같은 서버를 MCPAdapter 로 붙여 LangChain 도구 목록을 얻는다.
adapter_dir = new_workdir("nb03-adapter")
adapter = MCPAdapter(PythonStdioTransport(
    SERVER_SCRIPT,
    args=server_argv(adapter_dir / "sessions.sqlite3", adapter_dir / "evidence"),
    env={},                      # 서버에 키를 넘기지 않는다
    cwd=str(ROOT),
))
adapter_tools = await adapter.list_tools()

rows = []
for tool in adapter_tools:
    # tool.args 는 인자 이름 -> JSON Schema 조각의 사전이다.
    enums = {name: spec["enum"] for name, spec in tool.args.items() if "enum" in spec}
    hints = tool.metadata["mcp"]["tool"]["annotations"]
    rows.append({
        "도구": tool.name,
        "인자": ", ".join(tool.args),
        "enum": enums or "-",
        "idempotent": hints.get("idempotent_hint"),
        "read_only": hints.get("read_only_hint"),
    })
print("도구 객체의 종류:", type(adapter_tools[0]).__name__, "| 도구 수:", len(adapter_tools))
print("mcp.Client 의 순서와 같은가:", [tool.name for tool in adapter_tools] == first_names)
show_table(rows)
print("\nrecord_answer 의 설명(모델이 읽는 문장):", [t for t in adapter_tools if t.name == "record_answer"][0].description)

**관찰할 것.** 이름, 설명, 인자 이름, `field` 의 **enum 값**이 모두 그대로 남았습니다. 도구 순서도 같습니다.
MCP 의 `annotations` 는 LangChain 도구의 `metadata["mcp"]["tool"]["annotations"]` 에 남지만, 키 이름이
`idempotentHint` 에서 `idempotent_hint` 로 바뀌었습니다. 이 힌트는 **모델에게 직접 전달되는 것이 아니라**
도구 객체의 부가 정보로만 존재합니다. 모델이 읽는 것은 이름, 설명, 인자 스키마입니다.
그러므로 `record_answer` 의 enum 과 한 줄 설명이 모델의 도구 사용 정확도를 좌우합니다. 02 노트북이 다룬 주제입니다.

In [ ]:
# [따라하기] 어댑터 도구를 직접 한 번 실행한다. 모델이 도구를 부를 때 실행되는 바로 그 함수다.
# 1) 먼저 세션을 만든다. LangChain 도구의 coroutine 속성이 모델이 도구를 부를 때 실행되는 함수다.
by_name = {tool.name: tool for tool in adapter_tools}
content, artifact = await by_name["start_interview"].coroutine(goal="어댑터 관찰용 목표")

# 2) 결과는 (content, artifact) 한 쌍이다. response_format 이 content_and_artifact 이기 때문이다.
print("response_format:", by_name["start_interview"].response_format)
print("content 의 종류:", type(content).__name__, "| 첫 블록의 type:", content[0]["type"])
print("artifact 의 키:", sorted(artifact))
print("content 본문(앞 80자):", content[0]["text"][:80].replace("\n", " "), "...")
adapter_session = artifact["structured_content"]["session_id"]

# 3) 도구 오류는 예외가 된다. mcp.Client 에서는 isError=True 였던 것이 여기서는 예외로 바뀐다.
try:
    await by_name["record_answer"].coroutine(session_id=adapter_session, field="budget", answer="무엇이든")
except Exception as error:
    print("\n도구 오류 -> 예외:", type(error).__name__)
    print("   메시지 첫 줄:", str(error).splitlines()[0])
print("이 도구의 handle_tool_error 처리기:", by_name["record_answer"].handle_tool_error.__name__)

# 4) 어댑터가 연 서버 연결을 닫는다. 닫지 않으면 자식 프로세스가 남는다.
await adapter.client.close()

**관찰할 것.** 어댑터 도구는 `(content, artifact)` 한 쌍을 돌려줍니다. LangChain 은 이 중 `content`(텍스트 블록 목록)를
`ToolMessage` 의 본문으로 삼고 `artifact` 는 메시지에 첨부로만 둡니다(`.venv/lib/python3.13/site-packages/langchain_core/tools/base.py:547-552`의
`response_format` 설명). 즉 모델은 `structuredContent` 가 아니라 **본문 텍스트**를 읽습니다.
이 서버는 두 곳에 같은 JSON 을 담아 두었기 때문에 어느 쪽을 읽어도 같은 정보를 얻습니다.

`mcp.Client` 에서 `isError=True` 로 보였던 enum 위반이 여기서는 예외(`_MCPToolExecutionError`)가 되었습니다.
이 예외를 에이전트 쪽에서 어떻게 받아 모델에게 돌려주는지는 이 노트북이 실행으로 확인하지 않았습니다.
도구에 `handle_tool_error` 처리기가 달려 있다는 사실까지만 확인했습니다.

In [ ]:
# [바꿔보기] 같은 서버를 interface="plain" 으로 띄우면 모델이 보는 정의가 어떻게 달라지는가.
# 아래 INTERFACE 를 "rich" 로 바꾸면 위의 표와 같은 결과가 나온다. 서버가 하는 업무는 둘이 같다.
INTERFACE = "plain"

async with connect_server(new_workdir("nb03-plain"), interface=INTERFACE) as plain_client:
    plain_tools = {tool.name: tool for tool in (await plain_client.list_tools()).tools}

record_definition = plain_tools["record_answer"]
field_spec = record_definition.input_schema["properties"]["field"]
print(f"interface={INTERFACE!r} 의 record_answer 정의")
print("  설명:", record_definition.description)
print("  field 인자의 스키마:", field_spec)
print("  outputSchema 가 있는가:", record_definition.output_schema is not None)
print("  annotations 가 있는가:", record_definition.annotations is not None)

**관찰할 것.** `plain` 에서는 `field` 의 enum 이 사라지고 설명이 한 줄로 줄어듭니다(`outputSchema` 와 annotations 도 공개되지 않습니다).
서버가 하는 일은 같은데 모델이 읽는 정의만 약해졌습니다. 같은 요청에서 모델이 `field` 에 엉뚱한 값을 넣는 일이 얼마나 늘어나는지는
02 노트북이 작은 실측으로 비교합니다. 이 서버의 `--interface` 옵션은 그 비교를 위해 만든 것입니다(`cap_lab/server/server.py:182-196` 의 `publish`).

## 6. Deep Agent 한 턴: 모델이 도구를 고른다

이제 사람이 하던 일을 모델이 합니다. `cap_lab/agent.py:89` 의 `build_agent` 가 한 번에 다음을 조립합니다.

1. `MCPAdapter` 로 MCP 서버(자식 프로세스)를 띄우고 도구 7개를 얻는다.
2. 도구 7개에 `defer_loading` 을 켠다(`cap_lab/agent.py:148` 의 `_deferred`). **Skill 의 `SKILL.md` 를 읽기 전에는 모델에 도구 스키마가 가지 않습니다.**
   Skill 이 `metadata.include_tools` 에 도구 이름을 나열해 두었기 때문입니다(`skills/requirements-interview/SKILL.md:8`).
3. `create_deep_agent` 로 에이전트를 만든다. 파일 도구는 Skill 폴더만 읽을 수 있고 쓰기는 거절된다(`cap_lab/agent.py:137`).

그래서 첫 번째 도구 호출은 보통 `start_interview` 가 아니라 **`read_file` 로 `SKILL.md` 를 읽는 것**일 것입니다.
Skill 과 도구 공개의 관계는 05 노트북이 본격적으로 다룹니다. 여기서는 모델이 고른 도구 이름과 인자만 봅니다.

사용할 사례는 개발용(dev) 사례 `dev-01` 의 사용자 발화입니다. "비품 대여 현황 도구를 만들고 싶다. 어떤 걸 정해야 하는지 질문으로 정리해 달라"는 요청입니다.
이 노트북에서 본 서버의 `start_interview` 가 필요한 상황입니다.

**먼저 예측하세요.** 모델 호출은 몇 번, 도구 호출은 몇 번일까요? 호출되는 도구의 **순서**를 적어 보세요.
`record_answer` 나 `freeze_seed` 까지 갈까요? (사용자는 아직 제약도 완료 기준도 말하지 않았습니다.)

In [ ]:
# [따라하기] 도우미 정의만 한다. 함수를 만들 뿐이므로 모델도 서버도 부르지 않는다(호출은 아래 셀과 해보기 셀에서 일어난다).
announce("ask_agent 정의", model_calls=0)


async def ask_agent(case_id: str, label: str) -> dict:
    """dev 사례의 첫 발화를 에이전트에 한 번 보내고, 모델이 부른 도구들을 모아서 돌려준다.

    case_id 는 dev 사례만 받는다. 작업 폴더 label 이 새로 만들어지고 서버 자식 프로세스는 끝에 반드시 닫힌다.
    """
    # 1) 사례 찾기: dev 사례 목록에서만 찾는다.
    dev_cases = {case["id"]: case for case in load_cases(split="dev")}
    case = dev_cases[case_id]
    user_turn = case["turns"][0]

    # 2) 작업 폴더, 에이전트 조립. 키가 없으면 여기서 RuntimeError 로 멈추고 .env.example 을 안내한다.
    work = new_workdir(label)
    handle = await build_agent(
        load_settings(),
        server_args=server_argv(work / "sessions.sqlite3", work / "evidence"),
        skills_dir=SKILLS_DIR,
    )
    tool_calls, model_calls, skill_read, final_text = [], 0, False, ""
    try:
        deferred = {tool.name: bool((tool.extras or {}).get("defer_loading")) for tool in handle.tools}
        print("MCP 도구 7개의 defer_loading:", deferred)
        print("사용자 발화:", user_turn)

        # 3) 한 턴을 스트림으로 받는다. updates 모드는 노드(model, tools)가 끝날 때마다 한 묶음씩 온다.
        #    recursion_limit 은 모델이 같은 도구를 끝없이 부르는 일을 막는 안전장치다.
        request = {"messages": [{"role": "user", "content": user_turn}]}
        async for update in handle.agent.astream(request, {"recursion_limit": 50}, stream_mode="updates"):
            for node, output in update.items():
                if node == "model":
                    model_calls += 1
                    for message in output.get("messages", []):
                        for call in getattr(message, "tool_calls", []):
                            tool_calls.append(call["name"])
                            skill_read = skill_read or (call["name"] == "read_file" and "SKILL.md" in str(call["args"]))
                            print(f"  모델 호출 {model_calls}: 도구 {call['name']} 인자={json.dumps(call['args'], ensure_ascii=False)[:120]}")
                        if not getattr(message, "tool_calls", []):
                            final_text = message.text
                elif node == "tools":
                    for message in output.get("messages", []):
                        # 어댑터 도구의 결과는 텍스트 블록 목록이다. 첫 블록의 글자만 꺼내 보여 준다.
                        blocks = message.content if isinstance(message.content, list) else [message.content]
                        first_text = blocks[0]["text"] if blocks and isinstance(blocks[0], dict) else str(blocks[0])
                        print(f"  도구 결과: {message.name} -> {' '.join(first_text.split())[:90]!r}")
    finally:
        # 4) 어떤 경우에도 MCP 서버 자식 프로세스를 닫는다.
        await handle.aclose()
    return {"tools": tool_calls, "model_calls": model_calls, "skill_read": skill_read,
            "final": final_text, "workdir": work}


In [ ]:
# [따라하기] 유료 1회: 에이전트에 사용자 발화 한 번을 보내고, 모델이 부른 도구를 순서대로 본다.
announce("Deep Agent 한 턴(dev-01)", model_calls=6)


agent_run = await ask_agent("dev-01", "nb03-agent")
print("\n--- 요약 (1회 관찰) ---")
print("모델 호출 수:", agent_run["model_calls"])
print("Skill 을 읽었는가:", agent_run["skill_read"])
print("부른 도구 순서:", agent_run["tools"])
print("최종 답변(앞 300자):", agent_run["final"][:300])

**관찰할 것.** 이것은 **1회 관찰**입니다. 같은 발화를 다시 보내면 모델이 다른 도구 순서를 고를 수도 있습니다.
같은 설정에서도 결과가 달라질 수 있다는 것이 반복 실측이 필요한 이유이며, 06 노트북이 이를 다룹니다.
확인할 것은 세 가지입니다.

1. `defer_loading` 이 7개 모두 `True` 였는지. 그렇다면 `SKILL.md` 를 읽기 전에는 모델이 이 도구들의 스키마를 받지 못합니다.
2. 도구 호출 순서에 `read_file`(Skill 읽기)이 `start_interview` 보다 앞서는지.
3. `start_interview` 의 인자 `goal` 이 사용자 발화의 목표 문장과 얼마나 같은지. 모델이 목표를 바꿔 쓰면 이후 모든 단계가 어긋납니다.

그리고 방금 사람이 한 걸음마다 부른 도구를, 이번에는 모델이 부른 것입니다. 호출 한 번의 모양(`tools/call`, `_meta`, 결과)은 같습니다.
다른 것은 **누가 다음 도구를 고르느냐**뿐입니다.

## 직접 해보기

### 해보기 1. 일시 오류를 주입하고 다시 부르기 (무료)

서버는 `--fault` 로 장애를 주입할 수 있습니다(`cap_lab/server/server.py:64-71` 의 `FaultPlan`). 아래 셀은 `record_answer` 의 처음 한 번을
`temporarily_unavailable` 로 거절하게 합니다. 결과를 읽고, `FAULTS` 의 숫자나 도구 이름을 바꿔서 다시 실행하세요.
`permission_denied=["save_evidence"]` 로 바꾸면 무슨 일이 생길까요? 재시도해도 소용없는 오류와 재시도하면 되는 오류를
결과의 어떤 필드가 구분해 주는지 찾아보세요.

In [ ]:
# [바꿔보기] 일시 오류 주입. FAULTS 의 값을 바꿔 가며 같은 호출을 반복해 본다.
FAULTS = FaultPlan(transient={"record_answer": 1})

fault_dir = new_workdir("nb03-fault")
async with connect_server(fault_dir, fault=FAULTS) as client:
    fault_session = (await client.call_tool("start_interview", {"goal": "장애 주입 연습"})).structured_content["session_id"]
    for attempt in (1, 2):
        result = await client.call_tool(
            "record_answer", {"session_id": fault_session, "field": "constraints", "answer": "없음"}
        )
        body = result.structured_content
        print(f"{attempt}번째 호출: isError={result.is_error} success={body['success']} "
              f"error={body.get('error')} retryable={body.get('retryable')}")

### 해보기 2. 내 목표로 한 세션 걷기 (무료)

구간 3의 걷기를 본인이 고른 목표, 제약, 완료 기준으로 다시 하세요. 목표는 지금 맡은 일이나 만들고 싶은 도구면 좋습니다.
실행한 뒤 `show_events` 의 표에서 기록이 몇 개 쌓였는지, 상태가 언제 `ready` 가 되었는지 말로 설명해 보세요.

In [ ]:
# [내 것 만들기] 아래 세 줄을 내 것으로 바꾸고 실행한다.
MY_GOAL = "여기에 만들고 싶은 도구나 업무 결과를 한 문장으로 쓴다"
MY_CONSTRAINT = "여기에 지켜야 할 제약을 쓴다 (없으면 '없음')"
MY_CRITERION = "여기에 실행 결과로 참/거짓을 확인할 수 있는 완료 기준을 쓴다"

my_dir = new_workdir("nb03-mine")
async with connect_server(my_dir) as client:
    mine = await step(client, "내 목표", "start_interview", {"goal": MY_GOAL})
    for field, answer in (("constraints", MY_CONSTRAINT), ("acceptance_criteria", MY_CRITERION)):
        await step(client, f"내 답변 {field}", "record_answer",
                   {"session_id": mine["session_id"], "field": field, "answer": answer})
    mine_frozen = await step(client, "내 확정", "freeze_seed", {"session_id": mine["session_id"]})

print("확정한 명세:", json.dumps(mine_frozen["seed"], ensure_ascii=False, indent=2))
show_events(my_dir / "evidence")

### 해보기 3. 개념 질문에는 도구를 쓰지 않는가 (유료, 잠겨 있음)

`RUN_EXERCISE = True` 로 바꾸면 개발용 사례 `dev-02`(MCP가 뭔지 설명해 달라는 개념 질문)를 에이전트에 한 번 보냅니다.
구간 6의 `dev-01` 결과와 비교하세요. 이 질문에서 `read_file` 이나 `start_interview` 를 부르면 무엇이 문제일까요?
`ask_agent` 는 구간 6의 앞쪽 셀(모델을 부르지 않는 정의 셀)에서 만들어지므로, 구간 6의 유료 셀을 실행하지 않아도 이 셀은 동작합니다. 다만 이 해보기도 자기 유료 호출을 한 번 더 일으키므로 `RUN_EXERCISE = True` 로 켠 뒤에만 모델을 부릅니다.
마지막 `dev-01` 비교는 구간 6의 유료 셀이 만든 `agent_run` 을 읽습니다. 그 셀을 건너뛰었다면 비교 대신 안내 문장을 출력합니다.

In [ ]:
# [바꿔보기] 유료 잠금. True 로 바꾸면 모델 호출이 일어난다.
RUN_EXERCISE = False

if RUN_EXERCISE:
    announce("개념 질문 한 턴(dev-02)", model_calls=4)
    concept_run = await ask_agent("dev-02", "nb03-concept")
    print("\n--- 요약 (1회 관찰) ---")
    print("부른 도구 순서:", concept_run["tools"], "| Skill 을 읽었는가:", concept_run["skill_read"])
    if "agent_run" in globals():
        print("dev-01 과 비교:", agent_run["tools"])
    else:
        print("dev-01 과 비교: 구간 6의 유료 셀(셀 43)을 실행하지 않아 비교할 결과가 없습니다.")
else:
    print("RUN_EXERCISE 가 False 입니다. 모델을 부르지 않았습니다.")

### 해보기 4. 같은 발화를 3번 보내 흔들림 보기 (유료, 잠겨 있음, 선택)

`RUN_OPTIONAL = True` 로 바꾸면 `dev-01` 을 **3번** 반복합니다. 같은 발화에 모델이 같은 도구 순서를 고르는지 보세요.
3번이라도 표본이 작으므로 "대체로 같다" 이상의 결론은 내릴 수 없다는 점을 먼저 적어 두고 시작하세요.

In [ ]:
# [바꿔보기] 선택 실습 잠금. True 로 바꾸면 에이전트 턴을 3번 반복한다(Claude 약 18회 이하).
RUN_OPTIONAL = False
REPEATS = 3

if RUN_OPTIONAL:
    announce("dev-01 반복 관찰", model_calls=6 * REPEATS)
    repeat_runs = [await ask_agent("dev-01", f"nb03-rep{number}") for number in range(1, REPEATS + 1)]
    show_table([{"반복": number, "모델 호출": run["model_calls"], "Skill 읽음": run["skill_read"],
                 "도구 순서": " -> ".join(run["tools"])} for number, run in enumerate(repeat_runs, start=1)])
    print("서로 다른 도구 순서의 수:", len({tuple(run["tools"]) for run in repeat_runs}))
else:
    print("RUN_OPTIONAL 이 False 입니다. 모델을 부르지 않았습니다.")

## 확인 질문

1. stdio 서버에서 `print("시작")` 이 왜 위험한가요? 줄바꿈이 있을 때와 없을 때 클라이언트의 결과가 어떻게 달랐는지, 그 이유를 한 문장으로 설명하세요.
2. 2026-07-28 판에서 `initialize` 없이도 `tools/list` 를 보낼 수 있는 이유는 무엇인가요? 요청에서 `_meta` 를 빼면 어떻게 되었나요?
3. `freeze_seed` 가 `seed_incomplete` 로 거절된 경우와 `record_answer` 의 `field` 에 허용되지 않은 값을 넣은 경우를 `isError` 와 `structuredContent` 기준으로 구분해 설명하세요. 모델은 각각 어떻게 회복할 수 있나요?
4. `MCPAdapter` 를 거치면 MCP 도구의 이름, 설명, 인자 스키마, enum, annotations 중 모델에게 보이는 것과 도구 객체 안에만 남는 것은 각각 무엇인가요?
5. 구간 6의 결과가 1회 관찰이라는 말은 무엇을 뜻하나요? "이 에이전트는 항상 `read_file` 을 먼저 부른다"고 결론 내리려면 무엇이 더 필요한가요?

## 정리

- stdio 서버는 클라이언트가 띄운 자식 프로세스입니다. **stdout 은 프로토콜 전용, 진단은 stderr** 입니다(`cap_lab/server/server.py:435`). 어기면 응답이 사라지거나, 더 나쁘게는 경고만 남기고 동작합니다.
- 2026-07-28 판의 요청은 스스로 설명합니다. `initialize` 없이 `_meta`(프로토콜 버전, 클라이언트 정보, capabilities)를 실어 보내고, 첫 줄은 `server/discover` 입니다.
- 이 서버에서 `tools/list` 의 순서는 연결이 달라도 같았고 서버가 등록한 순서였습니다.
- 세션은 `draft -> ready -> frozen` 으로 움직이고 상태는 저장된 기록에서 다시 계산됩니다(`cap_lab/server/workflow.py:82`).
- 오류는 두 종류입니다. 업무 오류는 `isError: false` 와 `success: false` 와 오류 코드, 도구 오류(스키마 위반)는 `isError: true` 와 본문 텍스트입니다.
- `MCPAdapter` 는 이름, 설명, 인자 스키마(enum 포함)를 그대로 LangChain 도구로 옮기고, 결과를 `(content, artifact)` 한 쌍으로 돌려줍니다. 모델이 읽는 것은 본문 텍스트입니다.
- 모델이 도구를 고르는 것도 같은 `tools/call` 입니다. 다른 점은 다음 도구를 고르는 주체이고, 그 선택은 반복해서 재야 합니다. 이 노트북의 모델 결과는 1회 관찰입니다.